# Policy Gradient

**Policy Gradient** menandai pergeseran paradigma dalam Reinforcement Learning.

Jika pada **Q-Learning, DQN, dan SARSA** agen belajar memperkirakan **nilai aksi (Value-based)** lalu memilih aksi terbaik, pada **Policy Gradient** agen **langsung mengoptimalkan kebijakan (Policy-based)** tanpa perlu membuat Q-Table atau Q-Network.

## Mengapa Menggunakan Policy Gradient?

| Aspek                         | Value-Based (Q-Learning / DQN)             | Policy-Based (Policy Gradient)                                                  |
| ----------------------------- | ------------------------------------------ | ------------------------------------------------------------------------------- |
| **Fokus Utama**               | Memprediksi skor `Q(s, a)`                 | Memprediksi probabilitas aksi `πθ(a \| s)`                                      |
| **Ruang Aksi (Action Space)** | Cocok untuk aksi diskrit (`Atas`, `Bawah`) | Mendukung aksi **diskrit & kontinyu**, misalnya kemudi mobil `-45°` s.d. `+45°` |
| **Sifat Kebijakan**           | Deterministik (`argmax`)                   | Stokastik (probabilistik, mampu belajar strategi acak optimal)                  |
| **Stabilitas**                | Rentan terombang-ambing (*oscillation*)    | Perubahan *policy* terjadi secara lebih mulus (*smooth*)                        |

## Konsep Matematis & Inti Algoritma REINFORCE

Policy diparameterisasi oleh Neural Network dengan bobot `θ`.

Tujuan utama adalah memaksimalkan ekspektasi total reward `J(θ)`.

### Policy Gradient Theorem

$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\pi_\theta}
\left[
\nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R_t
\right]
$$

### Intuisi Sederhana

1. Agen menjalankan **1 episode penuh** dari awal sampai selesai dan mengumpulkan pasangan `(s_t, a_t, r_t)`.
2. Hitung **total reward kumulatif `R_t`** dari setiap langkah.
3. Jika `R_t` **positif/tinggi**, naikkan probabilitas `π(a_t | s_t)` agar aksi tersebut lebih sering dipilih di masa depan.
4. Jika `R_t` **negatif/rendah**, turunkan probabilitas `π(a_t | s_t)`.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as env_lib
import numpy as np

# 1. Definisi Policy Network (Menghasilkan Distribusi Probabilitas Aksi)
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        logits = self.fc(x)
        return F.softmax(logits, dim=-1)  # Output berupa probabilitas (total = 1.0)

# 2. Setup Environment & Hyperparameter
env = env_lib.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy_net = PolicyNetwork(state_dim, action_dim)
optimizer = optim.Adam(policy_net.parameters(), lr=1e-2)
gamma = 0.99
num_episodes = 500

# 3. Training Loop (REINFORCE)
for episode in range(num_episodes):
    state, _ = env.reset()
    log_probs = []
    rewards = []
    done = False

    # --- Step A: Jalankan 1 Episode Penuh ---
    while not done:
        state_t = torch.FloatTensor(state)
        probs = policy_net(state_t)

        # Sampling aksi berdasarkan distribusi probabilitas
        m = Categorical(probs)
        action = m.sample()

        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        log_probs.append(m.log_prob(action))
        rewards.append(reward)
        state = next_state

    # --- Step B: Hitung Return Kumulatif (R_t) ---
    discounted_returns = []
    R = 0
    for r in reversed(rewards):
        R = r + gamma * R
        discounted_returns.insert(0, R)

    discounted_returns = torch.tensor(discounted_returns)
    # Normalisasi returns untuk memperstabil training
    discounted_returns = (discounted_returns - discounted_returns.mean()) / (discounted_returns.std() + 1e-8)

    # --- Step C: Hitung Loss & Backpropagation ---
    policy_loss = []
    for log_prob, R_t in zip(log_probs, discounted_returns):
        policy_loss.append(-log_prob * R_t)  # Minus karena PyTorch melakukan Gradient Descent

    optimizer.zero_grad()
    policy_loss = torch.stack(policy_loss).sum()
    policy_loss.backward()
    optimizer.step()

    if (episode + 1) % 50 == 0:
        total_reward = sum(rewards)
        print(f"Episode {episode + 1}/{num_episodes} - Total Reward: {total_reward}")

env.close()

Episode 50/500 - Total Reward: 500.0
Episode 100/500 - Total Reward: 23.0
Episode 150/500 - Total Reward: 95.0
Episode 200/500 - Total Reward: 500.0
Episode 250/500 - Total Reward: 500.0
Episode 300/500 - Total Reward: 500.0
Episode 350/500 - Total Reward: 11.0
Episode 400/500 - Total Reward: 500.0
Episode 450/500 - Total Reward: 272.0
Episode 500/500 - Total Reward: 500.0
